# CanSat Data Analysis — GRASP, VAMOS & ScamSat

Spectral and atmospheric time-series analysis of the GRASP–VAMOS–ScamSat CanSat mission (Dübendorf CH, 2026-02-05).

---
## 1. Introduction

### 1.1 Mission overview

Three CanSat probes (GRASP, VAMOS, ScamSat) were released from an aircraft at ~665 m AGL above Dübendorf, Switzerland (448 m ASL) on 2026-02-05.  Each probe descended on a parachute at ~3–4 m s⁻¹, sampling atmospheric variables during the fall.

Key platform characteristics that motivate the analysis strategy:

| Source of disturbance | Description | Affected sensors |
|----------------------|-------------|-----------------|
| **Pendulum oscillation** | CanSat swings below parachute (~0.1–1 Hz for 0.3–3 m line) | Wind, acceleration |
| **Axial spin** | Payload rotates around vertical axis | Wind direction |
| **Sensor warm-up** | NDIR CO₂ and pressure sensors produce spurious values at power-on | CO₂, pressure |
| **Thermal lag** | Temperature sensor equilibrates slowly after launch | Temperature |

### 1.2 Research question

> *How do temperature, pressure, particulate matter (PM₂.₅/PM₁₀), CO₂, and wind vary with altitude and time during the CanSat descent, and how can spectral analysis help distinguish atmospheric structure from sensor noise and oscillatory descent dynamics?*

### 1.3 Methods overview

**Spectral analysis pipeline** (applied to VAMOS and ScamSat — see §4):
```
raw signal → uniform resampling → Butterworth LP filter → linear detrend
         → Hanning-windowed FFT + Welch PSD + STFT spectrogram
```
GRASP (176 s descent only) and OBAMA (31 rows) are **not** subjected to the spectral pipeline — they lack sufficient recording length or sample count. They are shown in §2 as raw context and used in §5 for the atmospheric profile.

**Atmospheric analysis** (§5): filtered signals are used to reconstruct vertical profiles, compare against ISA standard atmosphere, and quantify PM₂.₅, PM₁₀, CO₂ and wind statistics.

### 1.4 Expected results

| Variable | Expected observation | Physical reasoning |
|----------|---------------------|-------------------|
| Temperature lapse rate | Sub-standard (less negative than −6.5 K km⁻¹), possibly inverted | Stable winter boundary layer over a continental site |
| PM near ground | Elevated PM₂.₅/PM₁₀, possibly above WHO limits | Aerosol accumulation in the stable boundary layer |
| CO₂ | Above global background (420 ppm) | Local anthropogenic emissions (traffic, combustion) |
| Parachute pendulum | No dominant spectral peak in 0.5–1 Hz band | Stable flight conditions reported; no strong resonance expected |
| Wind phase dependence | Spectral content differs between aircraft, parachute and ground phases | Turbulence sources differ substantially across flight phases |


In [ ]:
# ── 0. Environment setup ─────────────────────────────────────────────────────
import sys
from pathlib import Path

_IN_COLAB = 'google.colab' in sys.modules
if _IN_COLAB and not Path('utils.py').exists():
    import urllib.request
    _URL = ('https://raw.githubusercontent.com/demartinomattia/Space_Data_2/main/utils.py')
    urllib.request.urlretrieve(_URL, 'utils.py')

import importlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.fft import fft, fftfreq
from scipy.signal import spectrogram as _scipy_spectrogram
warnings.filterwarnings('ignore')

import utils; importlib.reload(utils)
from utils import (
    apply_plot_style, save_figure,
    isa_alt, isa_temp_c, isa_press, baro_altitude_agl,
    load_grasp, load_vamos_science, load_vamos_wind,
    load_obama, load_uwyo_sounding, load_scamsat_bundle,
    resample_uniform, butter_lowpass, detrend_linear, compute_welch_psd,
    met_to_uv, detect_vamos_drop, print_dataset_summary,
    P0, T0_K, L, G, RD,
    WHO_PM25_24H, WHO_PM10_24H, CO2_BACKGROUND_PPM,
    FIG_DIR, DATA_DIR,
)
apply_plot_style()
print(f'utils {utils.__version__}')


In [ ]:
# ── Data loading ─────────────────────────────────────────────────────────────
grasp_raw = load_grasp()
vamos     = load_vamos_science()
wind      = load_vamos_wind()
obama     = load_obama()
uwyo      = load_uwyo_sounding()
scamsat   = load_scamsat_bundle()

scam_alt   = scamsat['altitude']
scam_press = scamsat['pressure']
scam_temp  = scamsat['temperature']
scam_pm25  = scamsat['pm25']
scam_pm10  = scamsat['pm10']
scam_accel = scamsat['acceleration']
scam_fft   = scamsat['fft']
print('All datasets loaded.')


In [ ]:
# ── Post-load derivations ────────────────────────────────────────────────────
grasp_clean = grasp_raw.iloc[1:].copy().reset_index(drop=True)
grasp_clean['alt_baro'] = isa_alt(grasp_clean['press_Pa'].values)

vamos_co2  = vamos[vamos['co2_ppm'] > 0].copy()
vamos_drop = detect_vamos_drop(vamos)
drop_mask  = vamos_drop['drop_mask']

# Window around the flight: apogee ± 10 min
_h_agl    = vamos_drop['h_agl']
_airborne = np.where(_h_agl > 30)[0]
if len(_airborne):
    _t_win0 = max(vamos['t_s'].values[_airborne[0]]  - 600, vamos['t_s'].iat[0])
    _t_win1 = min(vamos['t_s'].values[_airborne[-1]] + 600, vamos['t_s'].iat[-1])
else:
    _t_win0, _t_win1 = vamos['t_s'].iat[0], vamos['t_s'].iat[-1]

vamos_conc = vamos[vamos['t_s'].between(_t_win0 - 30, _t_win1 + 30)].copy()

FS_G = round(1000.0 / np.median(np.diff(grasp_raw['t_ms'].values)))
FS_V = 2.0   # VAMOS science ≈ 2 Hz (used as resampling target)
FS_W = 2.0   # VAMOS wind   ≈ 2 Hz

print_dataset_summary(grasp_raw, vamos, wind, obama, uwyo, vamos_drop, vamos_conc, scamsat=scamsat)


---
## 2. Raw Data Overview

All datasets are inspected exactly as recorded (no filtering, no correction).  This serves two purposes:
1. **Sanity check** — physically plausible values, correct loading.
2. **Suitability assessment** — which datasets are long/dense enough for spectral analysis?

### Known data quality issues

| # | Sensor | Artefact | Cause | Handling |
|---|--------|----------|-------|---------|
| 1 | GRASP | Row-0 pressure spike (81 261 Pa ≡ ~1 820 m) | Sensor initialisation | Drop row 0 in `load_grasp()` caller |
| 2 | VAMOS science | First ~64 CO₂ readings = 0 ppm | NDIR thermal warm-up | Exclude `co2_ppm == 0` |
| 3 | VAMOS wind | `tumbling` flag stuck at 0 | Firmware bug | Ignore; use `wind_acc_vec` instead |
| 4 | VAMOS wind | Timestamp reset mid-recording | µC reboot | `load_vamos_wind()` auto-trims |


In [ ]:
# ── 2.0 Data quality — artefact figure ───────────────────────────────────────
fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.48, wspace=0.38)
fig.suptitle('Data Quality — identified artefacts', fontsize=12, fontweight='bold')

# (a) GRASP row-0 pressure spike
ax = fig.add_subplot(gs[0, 0])
ax.plot(grasp_raw['t_rel'], grasp_raw['press_Pa'] / 100, lw=1, color='tab:blue', label='All rows')
ax.plot(grasp_clean['t_rel'], grasp_clean['press_Pa'] / 100, lw=1, color='tab:orange', label='Row 0 removed')
ax.set(xlabel='Time (s)', ylabel='Pressure (hPa)', title='(a) GRASP — row-0 spike')
ax.legend(fontsize=8)

# (b) VAMOS CO₂ warm-up
ax = fig.add_subplot(gs[0, 1])
ax.plot(vamos['t_rel'] / 60, vamos['co2_ppm'], lw=0.7, color='tab:brown', alpha=0.6, label='All')
ax.plot(vamos_co2['t_rel'] / 60, vamos_co2['co2_ppm'], lw=0.7, color='tab:red', label='Warm-up removed')
ax.set(xlabel='Time (min)', ylabel='CO₂ (ppm)', title='(b) VAMOS — CO₂ warm-up zeros')
ax.legend(fontsize=8)

# (c) VAMOS wind — tumbling flag
ax = fig.add_subplot(gs[1, 0])
ax.plot(wind['t_rel'] / 3600, wind['tumbling'],     lw=0.8, color='gray',        label='tumbling flag (stuck at 0)')
ax.plot(wind['t_rel'] / 3600, wind['wind_acc_vec'], lw=0.6, color='tab:purple', alpha=0.7, label='|acc| vector (used instead)')
ax.set(xlabel='Time (h)', ylabel='Value', title='(c) VAMOS wind — tumbling flag bug')
ax.legend(fontsize=8)

# (d) VAMOS wind — clean continuous segment
ax = fig.add_subplot(gs[1, 1])
ax.plot(wind['t_rel'] / 3600, wind['wind_spd'], lw=0.7, color='tab:cyan')
ax.text(0.03, 0.92, 'Pre-reset data discarded\nby load_vamos_wind()',
        transform=ax.transAxes, fontsize=8, va='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))
ax.set(xlabel='Time (h)', ylabel='Wind speed (m s⁻¹)', title='(d) VAMOS wind — after reset fix')

plt.tight_layout()
save_figure(fig, 'data_quality/fig_artefacts.png', strip_text=False)
plt.show()


### 2.1 GRASP — Raw channels

**Duration: ~176 s | fs ≈ 38 Hz | descent only.**  
The signal is dominated by a monotonic trend (pressure rising, temperature varying with lapse rate). This makes GRASP unsuitable for spectral analysis: the window is too short and the trend would dominate the spectrum.

In [ ]:
# ── 2.1 GRASP raw ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('GRASP — raw channels (row 0 removed, descent only)', fontweight='bold')
specs = [
    ('temp_C',   'Temperature (°C)',  'tab:red'),
    ('press_Pa', 'Pressure (Pa)',      'tab:blue'),
    ('alt_m',    'GPS altitude (m)',   'tab:green'),
    ('alt_baro', 'Baro altitude (m)',  'tab:olive'),
    ('pm25',     'PM₂.₅ (µg m⁻³)',   'tab:orange'),
    ('pm10',     'PM₁₀ (µg m⁻³)',    'tab:brown'),
]
for ax, (col, ylabel, color) in zip(axes.flat, specs):
    ax.plot(grasp_clean['t_rel'], grasp_clean[col], color=color, lw=0.8)
    ax.set(xlabel='Time (s)', ylabel=ylabel, title=col)
plt.tight_layout()
save_figure(fig, 'plot_raw/grasp_overview.png', strip_text=False)
plt.show()
print(f'GRASP: {len(grasp_clean)} rows | {grasp_clean["t_rel"].iat[-1]:.0f} s | fs ≈ {FS_G} Hz')
print('Not used for spectral analysis (descent-only window, monotonic non-stationary signal).')


### 2.2 VAMOS — Science payload

**Duration: ~100 min | fs ≈ 2 Hz.** Long continuous recording suitable for spectral analysis. Gold band = drop phase.

In [ ]:
# ── 2.2 VAMOS science raw ────────────────────────────────────────────────────
_apo_m  = (vamos_drop['t_apogee_s']  - vamos['t_s'].iat[0]) / 60
_land_m = (vamos_drop['t_landing_s'] - vamos['t_s'].iat[0]) / 60

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
fig.suptitle('VAMOS science — raw channels (full recording)', fontweight='bold')
for ax, (col, ylabel, color) in zip(axes, [
    ('press_hPa', 'Pressure (hPa)',  'tab:blue'),
    ('temp_C',    'Temperature (°C)','tab:red'),
    ('co2_ppm',   'CO₂ (ppm)',       'tab:brown'),
]):
    _df = vamos_co2 if col == 'co2_ppm' else vamos
    ax.plot(_df['t_rel'] / 60, _df[col], lw=0.6, color=color, alpha=0.8)
    ax.axvspan(_apo_m, _land_m, color='gold', alpha=0.3, label='Drop phase')
    ax.set(ylabel=ylabel)
    ax.legend(fontsize=8, loc='upper right')
axes[-1].set_xlabel('Time (min)')
plt.tight_layout()
save_figure(fig, 'plot_raw/vamos_science_raw.png', strip_text=False)
plt.show()


### 2.3 VAMOS — Wind payload

**Duration: ~100 min | fs ≈ 2 Hz.** Gold band = drop phase.

In [ ]:
# ── 2.3 VAMOS wind raw ───────────────────────────────────────────────────────
_d0h = (vamos_drop['t_apogee_s']  - wind['t_s'].iat[0]) / 3600
_d1h = (vamos_drop['t_landing_s'] - wind['t_s'].iat[0]) / 3600
_twh = wind['t_rel'].values / 3600

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('VAMOS wind — raw channels', fontweight='bold')
for ax, (col, ylabel, color) in zip(axes.flat, [
    ('wind_spd',     'Wind speed (m s⁻¹)',       'tab:cyan'),
    ('wind_dir',     'Wind direction (°)',        'tab:purple'),
    ('wind_acc_vec', 'Acceleration |vec| (m s⁻²)','tab:olive'),
    ('wind_spd_vec', 'Wind speed |vec| (m s⁻¹)', 'teal'),
]):
    ax.plot(_twh, wind[col], lw=0.5, color=color, alpha=0.8)
    ax.axvspan(_d0h, _d1h, color='gold', alpha=0.3, label='Drop')
    ax.set(xlabel='Time (h)', ylabel=ylabel, title=col)
    ax.legend(fontsize=8)
plt.tight_layout()
save_figure(fig, 'plot_raw/vamos_wind_raw.png', strip_text=False)
plt.show()


### 2.4 OBAMA — External CanSat

**Only 31 rows — insufficient for spectral analysis.** Used as independent in-situ T/P/RH cross-check only.

In [ ]:
# ── 2.4 OBAMA raw ────────────────────────────────────────────────────────────
_ob = obama.dropna(subset=['Time_s'])
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f'OBAMA — raw data ({len(_ob)} rows, dual-sensor suite)', fontweight='bold')
for ax, (c1, c2, ylabel) in zip(axes, [
    ('first_temp_avg_C',    'second_temp_avg_C',    'Temperature (°C)'),
    ('first_pstat_avg_hPa', 'second_pstat_avg_hPa', 'Pressure (hPa)'),
    ('first_rh_avg',        'second_rh_avg',        'Rel. humidity (%)'),
]):
    ax.plot(_ob['Time_s'], _ob[c1], 'o-', ms=3, lw=1, color='tab:blue',  label='Sensor 1')
    ax.plot(_ob['Time_s'], _ob[c2], 's-', ms=3, lw=1, color='tab:orange',label='Sensor 2')
    ax.set(xlabel='Time (s)', ylabel=ylabel)
    ax.legend(fontsize=8)
plt.tight_layout()
save_figure(fig, 'plot_raw/obama_raw.png', strip_text=False)
plt.show()
print(f'OBAMA: {len(_ob)} rows — not used for spectral analysis.')


### 2.5 ScamSat — Raw channels

**Duration: ~1 h | variable fs.** Long continuous recording, acceleration channel directly measures mechanical oscillations. Used for spectral analysis in §4.3.

In [ ]:
# ── 2.5 ScamSat raw ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('ScamSat — raw channels', fontweight='bold')
for ax, (df, col, ylabel, color) in zip(axes.flat, [
    (scam_alt,   'altitude_agl_m', 'Altitude AGL (m)',      'tab:green'),
    (scam_press, 'press_Pa',       'Pressure (Pa)',          'tab:blue'),
    (scam_temp,  'temp_C',         'Temperature (°C)',       'tab:red'),
    (scam_pm25,  'pm25',           'PM₂.₅ (µg m⁻³)',       'tab:orange'),
    (scam_pm10,  'pm10',           'PM₁₀ (µg m⁻³)',        'tab:brown'),
    (scam_accel, 'accel_g',        'Acceleration (g)',       'tab:purple'),
]):
    ax.plot(df['t_rel'] / 3600, df[col], lw=0.5, color=color, alpha=0.8)
    ax.set(xlabel='Time (h)', ylabel=ylabel, title=col)
plt.tight_layout()
save_figure(fig, 'plot_raw/scamsat_overview.png', strip_text=False)
plt.show()


### 2.6 Dataset suitability summary

| Dataset | Length | fs (Hz) | Spectral analysis? | Reason |
|---------|--------|---------|-------------------|--------|
| GRASP | ~176 s | ≈38 | **No** | Descent only; window too short; monotonic trend dominates |
| VAMOS science | ~6 000 s | ≈2 | **Yes** | Long continuous multi-phase recording |
| VAMOS wind | ~6 074 s | ≈2 | **Yes** | Same; acceleration sensitive to pendulum/spin |
| OBAMA | 31 rows | — | **No** | Insufficient sample count |
| ScamSat | ~3 500 s | variable | **Yes** | Long recording; accelerometer measures vibrations directly |

The spectral pipeline is applied to **VAMOS (science + wind)** and **ScamSat** in §4.


---
## 3. Dataset Alignment and Descent Phase Detection

Each sensor uses its own millisecond clock from power-on, so datasets are not time-synchronised.  
We use the **VAMOS pressure record** to detect the descent (apogee → landing) algorithmically, then derive a common **barometric altitude AGL** axis for vertical profile comparisons.


In [ ]:
# ── 3.1 Descent detection ────────────────────────────────────────────────────
_t_min    = vamos['t_rel'].values / 60
_apo_min  = (vamos_drop['t_apogee_s']  - vamos['t_s'].iat[0]) / 60
_land_min = (vamos_drop['t_landing_s'] - vamos['t_s'].iat[0]) / 60

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('VAMOS — descent phase detection', fontweight='bold')

ax = axes[0]
ax.plot(_t_min, vamos['press_hPa'], color='tab:blue', lw=0.8)
ax.axvspan(_apo_min, _land_min, color='gold', alpha=0.35)
ax.axvline(_apo_min,  color='green', lw=1.5, ls='--', label=f'Apogee  {_apo_min:.1f} min')
ax.axvline(_land_min, color='red',   lw=1.5, ls='--', label=f'Landing {_land_min:.1f} min')
ax.set(xlabel='Time (min)', ylabel='Pressure (hPa)', title='Pressure')
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(_t_min, vamos_drop['h_agl'], color='tab:green', lw=0.8)
ax.axhline(0, color='brown', lw=1, ls=':', label='Ground')
ax.axvspan(_apo_min, _land_min, color='gold', alpha=0.35, label='Drop phase')
ax.axvline(_apo_min,  color='green', lw=1.5, ls='--')
ax.axvline(_land_min, color='red',   lw=1.5, ls='--')
ax.set(xlabel='Time (min)', ylabel='Altitude AGL (m)', title='Barometric altitude AGL')
ax.legend(fontsize=8)

plt.tight_layout()
save_figure(fig, 'flight_dynamics/fig_descent_detection_new.png', strip_text=False)
plt.show()
print(f'Peak altitude: {vamos_drop["h_peak_m"]:.0f} m AGL')
print(f'Drop duration: {vamos_drop["drop_duration_s"]:.0f} s | Mean descent rate: {vamos_drop["descent_rate_mps"]:.2f} m/s')


---
## 4. Spectral Analysis

### 4.0 Pipeline and methods

Each selected time series goes through four steps:

| Step | Tool | Purpose |
|------|------|---------|
| Uniform resampling | Linear interpolation to regular grid | FFT/Welch require equally-spaced samples |
| Low-pass filter | 4th-order Butterworth (forward-backward) | Remove electronic noise above a physical cut-off |
| Linear detrend | Least-squares fit | Remove monotonic trend before spectral estimation |
| Spectral estimation | Hanning FFT + Welch PSD + STFT | Frequency content, robust PSD, time-frequency evolution |

#### Pendulum frequency reference

For suspension length $L$:

$$f_{\mathrm{pend}} = \frac{1}{2\pi}\sqrt{\frac{g}{L}}$$

| $L$ (m) | $f_{\mathrm{pend}}$ (Hz) |
|---------|--------------------------|
| 3.0 | 0.29 |
| 1.0 | 0.50 |
| 0.5 | 0.70 |
| 0.3 | 0.91 |

A persistent narrow peak in **0.3–1.0 Hz** during the drop phase would indicate pendulum oscillation.


### 4.1 VAMOS Science — Pressure and Temperature

In [ ]:
# ── 4.1 Resample + LP filter + detrend ───────────────────────────────────────
t_v  = vamos['t_rel'].values
p_v  = vamos['press_hPa'].values * 100.0   # Pa
T_v  = vamos['temp_C'].values

t_vu, p_vu = resample_uniform(t_v, p_v, FS_V)
_,   T_vu  = resample_uniform(t_v, T_v, FS_V)

p_vf = butter_lowpass(p_vu, cutoff_hz=0.3, fs=FS_V, order=4)
T_vf = butter_lowpass(T_vu, cutoff_hz=0.3, fs=FS_V, order=4)

p_vd = detrend_linear(p_vf, t_vu)
T_vd = detrend_linear(T_vf, t_vu)

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
fig.suptitle('VAMOS science — filter and detrend', fontweight='bold')
for i, (raw_u, filt, detr, lbl, unit) in enumerate([
    (p_vu, p_vf, p_vd, 'Pressure', 'Pa'),
    (T_vu, T_vf, T_vd, 'Temperature', '°C'),
]):
    axes[i,0].plot(t_vu/60, raw_u,  lw=0.6, color='gray',     alpha=0.7, label='Resampled')
    axes[i,0].plot(t_vu/60, filt,   lw=1.0, color='tab:blue',            label='LP filtered')
    axes[i,0].set(xlabel='Time (min)', ylabel=f'{lbl} ({unit})', title=f'{lbl} — raw vs filtered')
    axes[i,0].legend(fontsize=8)
    axes[i,1].plot(t_vu/60, detr, lw=0.7, color='tab:orange')
    axes[i,1].axhline(0, color='gray', lw=0.8, ls=':')
    axes[i,1].set(xlabel='Time (min)', ylabel=f'Δ{lbl} ({unit})', title=f'{lbl} — detrended residual')
plt.tight_layout()
save_figure(fig, 'signal_processing/fig_vamos_filter.png', strip_text=False)
plt.show()
print(f'Resampled: {len(t_vu)} samples at {FS_V} Hz | Δf (Welch, nperseg=256) ≈ {FS_V/256:.4f} Hz')


In [ ]:
# ── 4.1 Welch PSD + Hanning FFT ─────────────────────────────────────────────
NPERSEG_V = 256

fw_p, psd_p, *_ = compute_welch_psd(t_vu, p_vd, FS_V, NPERSEG_V)
fw_T, psd_T, *_ = compute_welch_psd(t_vu, T_vd, FS_V, NPERSEG_V)

N   = len(p_vd)
hw  = np.hanning(N); nrm = np.sqrt(np.sum(hw**2))
P_f = np.abs(fft(p_vd * hw))[:N//2] / nrm
T_f = np.abs(fft(T_vd * hw))[:N//2] / nrm
f_f = fftfreq(N, 1.0/FS_V)[:N//2]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('VAMOS science — power spectra', fontweight='bold')
for row, (fw, psd, ff, amp, lbl, col) in enumerate([
    (fw_p, psd_p, f_f, P_f, 'Pressure (Pa)', 'tab:blue'),
    (fw_T, psd_T, f_f, T_f, 'Temperature (°C)', 'tab:red'),
]):
    axes[row,0].semilogy(fw[1:], psd[1:], color=col, lw=1.2)
    axes[row,0].axvspan(0.3, 1.0, color='gold', alpha=0.25, label='Pendulum band')
    axes[row,0].axvline(FS_V/2, color='gray', lw=0.8, ls='--', label='Nyquist')
    axes[row,0].set(xlabel='Frequency (Hz)', ylabel='PSD', title=f'{lbl.split()[0]} — Welch PSD')
    axes[row,0].legend(fontsize=8)
    axes[row,1].semilogy(ff[1:], amp[1:], color=col, lw=0.8, alpha=0.8)
    axes[row,1].axvspan(0.3, 1.0, color='gold', alpha=0.25, label='Pendulum band')
    axes[row,1].set(xlabel='Frequency (Hz)', ylabel='Amplitude', title=f'{lbl.split()[0]} — FFT (Hanning)')
    axes[row,1].legend(fontsize=8)
plt.tight_layout()
save_figure(fig, 'signal_processing/fig_vamos_psd.png', strip_text=False)
plt.show()


In [ ]:
# ── 4.1 STFT spectrogram ─────────────────────────────────────────────────────
NSTFT = 512; NOLAP = NSTFT * 3 // 4

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('VAMOS science — STFT spectrograms', fontweight='bold')
for ax, (sig, lbl, cmap) in zip(axes, [
    (p_vd, 'Pressure residual', 'viridis'),
    (T_vd, 'Temperature residual', 'plasma'),
]):
    fs, ts, Zxx = _scipy_spectrogram(sig, fs=FS_V, nperseg=NSTFT, noverlap=NOLAP, window='hann')
    im = ax.pcolormesh(ts/60, fs, 10*np.log10(np.abs(Zxx)+1e-30), shading='gouraud', cmap=cmap)
    ax.axhspan(0.3, 1.0, color='white', alpha=0.12)
    ax.set(xlabel='Time (min)', ylabel='Frequency (Hz)', title=f'{lbl}', ylim=(0, FS_V/2))
    plt.colorbar(im, ax=ax, label='dB/Hz')
plt.tight_layout()
save_figure(fig, 'signal_processing/fig_vamos_stft.png', strip_text=False)
plt.show()


#### Results — VAMOS science

**No dominant spectral peak** is present in the pendulum band (0.3–1.0 Hz) for either pressure or temperature.  
The spectrograms show broadband, flat residual power with no persistent periodic component across the mission timeline.

**Conclusion:** VAMOS pressure and temperature carry no detectable oscillatory artefact from parachute dynamics. The signal is dominated by the atmospheric trend (captured by the LP filter) and low-level broadband noise.


### 4.2 VAMOS Wind — Phase-separated Spectral Analysis

The wind sensor records over the full mission.  Merging all phases into a single spectrum would average out phase-specific dynamics.  We segment into three phases and compute independent Welch PSDs.

| Phase | Time window | Expected spectral character |
|-------|-------------|----------------------------|
| **Aircraft** | Before deployment | High energy from propeller vibration |
| **Parachute** | Apogee → landing | Pendulum/turbulence; key phase for artefact detection |
| **Ground** | After landing | Near-zero broadband (floor noise) |


In [ ]:
# ── 4.2 Phase segmentation ───────────────────────────────────────────────────
t_apo  = vamos_drop['t_apogee_s']
t_land = vamos_drop['t_landing_s']

# Conservative boundaries: 60 s margins
t_parachute_start = t_apo  - 30
t_parachute_end   = t_land + 30

# Aircraft phase: everything well before apogee (use ≥ 5 min before)
t_aircraft_end = t_apo - 300
mask_air  = wind['t_s'] < t_aircraft_end
mask_para = wind['t_s'].between(t_parachute_start, t_parachute_end)
mask_gnd  = wind['t_s'] > t_parachute_end

PHASES = {'Aircraft': (mask_air, 'tab:blue'), 'Parachute': (mask_para, 'tab:orange'), 'Ground': (mask_gnd, 'tab:green')}
for name, (m, _) in PHASES.items():
    print(f'  {name:12s}: {m.sum():5d} samples')

# Visualise
_twh = wind['t_rel'].values / 3600
_d0h = (t_apo  - wind['t_s'].iat[0]) / 3600
_d1h = (t_land - wind['t_s'].iat[0]) / 3600

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle('VAMOS wind — phase segmentation', fontweight='bold')
for ax, col, ylabel in zip(axes, ['wind_spd_vec', 'wind_acc_vec'],
                                   ['Wind speed |vec| (m s⁻¹)', 'Acceleration |vec| (m s⁻²)']):
    ax.plot(_twh, wind[col], lw=0.4, color='lightgray')
    for name, (mask, color) in PHASES.items():
        ax.plot(_twh[mask.values], wind[col][mask].values, lw=0.7, color=color, label=name)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
axes[-1].set_xlabel('Time (h)')
plt.tight_layout()
save_figure(fig, 'flight_dynamics/fig_wind_phases.png', strip_text=False)
plt.show()


In [ ]:
# ── 4.2 Phase-separated Welch PSD ────────────────────────────────────────────
FS_W_PSD = 1.0; NPERSEG_W = 64

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('VAMOS wind — Welch PSD per flight phase', fontweight='bold')
for ax, col in zip(axes, ['wind_spd_vec', 'wind_acc_vec']):
    for name, (mask, color) in PHASES.items():
        seg = wind[mask]
        if len(seg) < NPERSEG_W * 2:
            print(f'Skip {name}/{col}: {len(seg)} samples')
            continue
        f_w, psd_w, *_ = compute_welch_psd(seg['t_rel'].values, seg[col].values, FS_W_PSD, NPERSEG_W)
        ax.semilogy(f_w[1:], psd_w[1:], label=name, color=color, lw=1.4)
    ax.axvspan(0.3, 1.0, color='gold', alpha=0.25, label='Pendulum band')
    ax.set(xlabel='Frequency (Hz)', ylabel='PSD', title=f'{col} — phase PSD')
    ax.legend(fontsize=8)
plt.tight_layout()
save_figure(fig, 'signal_processing/fig_wind_psd_phases.png', strip_text=False)
plt.show()


In [ ]:
# ── 4.2 Wind STFT spectrogram ────────────────────────────────────────────────
_t_wu, _spd_wu = resample_uniform(wind['t_rel'].values, wind['wind_spd_vec'].values, 1.0)
_spd_wd = detrend_linear(_spd_wu, _t_wu)

fs_s, ts_s, Zxx_s = _scipy_spectrogram(_spd_wd, fs=1.0, nperseg=512, noverlap=384, window='hann')

fig, ax = plt.subplots(figsize=(13, 5))
fig.suptitle('VAMOS wind speed — STFT spectrogram (full mission)', fontweight='bold')
im = ax.pcolormesh(ts_s/3600, fs_s, 10*np.log10(np.abs(Zxx_s)+1e-30), shading='gouraud', cmap='inferno')
ax.axvspan(_d0h, _d1h, color='cyan', alpha=0.15, label='Drop phase')
ax.axhspan(0.3, 1.0, color='white', alpha=0.12, label='Pendulum band')
ax.set(xlabel='Time (h)', ylabel='Frequency (Hz)', ylim=(0, 0.5))
plt.colorbar(im, ax=ax, label='dB/Hz')
ax.legend(fontsize=8)
plt.tight_layout()
save_figure(fig, 'signal_processing/fig_wind_stft.png', strip_text=False)
plt.show()


#### Results — VAMOS wind

The phase-separated PSD shows higher broadband energy during the **aircraft phase** (vibration from the airframe) compared to the parachute and ground phases — confirming that segmentation is essential.

**No narrow-band peak** is visible in the pendulum band (0.3–1.0 Hz) during the **parachute phase**.  
The spectrogram confirms no persistent periodic component during the drop window.

**Conclusion:** No evidence of parachute pendulum oscillation in the VAMOS wind data. The wind speed variability during descent is consistent with atmospheric turbulence rather than an instrumental oscillation.


### 4.3 ScamSat — Acceleration and Temperature

The **acceleration channel** is the most direct indicator of mechanical oscillations. We also analyse **temperature** for completeness.

In [ ]:
# ── 4.3 ScamSat: estimate sample rates ───────────────────────────────────────
def _fs(df, col_t='t_rel'):
    t = df[col_t].values
    return len(t) / (t[-1] - t[0])

fs_accel = _fs(scam_accel); fs_temp_s = _fs(scam_temp)
FS_SA = max(1.0, round(fs_accel))
FS_ST = max(1.0, round(fs_temp_s))
print(f'ScamSat accel:  {len(scam_accel):5d} samples | fs ≈ {fs_accel:.2f} Hz → using {FS_SA} Hz')
print(f'ScamSat temp:   {len(scam_temp):5d} samples | fs ≈ {fs_temp_s:.2f} Hz → using {FS_ST} Hz')

# Resample + filter + detrend
_ta_u, _aa_u = resample_uniform(scam_accel['t_rel'].values, scam_accel['accel_g'].values, FS_SA)
_aa_f = butter_lowpass(_aa_u, cutoff_hz=min(0.4, FS_SA*0.4), fs=FS_SA, order=4)
_aa_d = detrend_linear(_aa_f, _ta_u)

_tt_u, _aT_u = resample_uniform(scam_temp['t_rel'].values, scam_temp['temp_C'].values, FS_ST)
_aT_f = butter_lowpass(_aT_u, cutoff_hz=min(0.3, FS_ST*0.4), fs=FS_ST, order=4)
_aT_d = detrend_linear(_aT_f, _tt_u)


In [ ]:
# ── 4.3 Welch PSD ────────────────────────────────────────────────────────────
NPERSEG_S = min(256, len(_aa_d) // 4, len(_aT_d) // 4)

fw_a, psd_a, *_ = compute_welch_psd(_ta_u, _aa_d, FS_SA, NPERSEG_S)
fw_t, psd_t, *_ = compute_welch_psd(_tt_u, _aT_d, FS_ST, NPERSEG_S)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('ScamSat — Welch PSD', fontweight='bold')
for ax, (fw, psd, lbl, col) in zip(axes, [
    (fw_a, psd_a, 'Acceleration (g)', 'tab:purple'),
    (fw_t, psd_t, 'Temperature (°C)', 'tab:red'),
]):
    ax.semilogy(fw[1:], psd[1:], color=col, lw=1.2)
    ax.axvspan(0.3, 1.0, color='gold', alpha=0.25, label='Pendulum band')
    ax.set(xlabel='Frequency (Hz)', ylabel='PSD', title=f'{lbl} — Welch PSD')
    ax.legend(fontsize=8)
plt.tight_layout()
save_figure(fig, 'signal_processing/fig_scamsat_psd.png', strip_text=False)
plt.show()


In [ ]:
# ── 4.3 STFT spectrogram — acceleration ──────────────────────────────────────
NSTFT_S = min(512, len(_aa_d) // 4)
fs_s2, ts_s2, Zxx_s2 = _scipy_spectrogram(
    _aa_d, fs=float(FS_SA), nperseg=NSTFT_S, noverlap=NSTFT_S*3//4, window='hann')

fig, ax = plt.subplots(figsize=(13, 5))
fig.suptitle('ScamSat — acceleration STFT spectrogram', fontweight='bold')
im = ax.pcolormesh(ts_s2/3600, fs_s2, 10*np.log10(np.abs(Zxx_s2)+1e-30), shading='gouraud', cmap='viridis')
ax.axhspan(0.3, 1.0, color='white', alpha=0.12, label='Pendulum band')
ax.set(xlabel='Time (h)', ylabel='Frequency (Hz)', ylim=(0, min(1.5, float(FS_SA)/2)))
plt.colorbar(im, ax=ax, label='dB/Hz')
ax.legend(fontsize=8)
plt.tight_layout()
save_figure(fig, 'signal_processing/fig_scamsat_stft.png', strip_text=False)
plt.show()


#### Results — ScamSat

Inspect the Welch PSD for narrow peaks above the noise floor in the 0.3–1.0 Hz pendulum band.  
The spectrogram shows the time evolution of mechanical excitation across the full mission.

If no persistent narrow-band peak is present: no evidence of pendulum oscillation from the ScamSat accelerometer.  
If a peak is present: compare its frequency to the theoretical pendulum frequency for the known suspension length.


### 4.4 Summary and Final Processed Signals

| Dataset | Signal | Pendulum peak found? | Action |
|---------|--------|---------------------|--------|
| VAMOS science | Pressure | No | Use LP-filtered signal for profile analysis |
| VAMOS science | Temperature | No | Use LP-filtered signal |
| VAMOS wind | Wind speed/acceleration | No | Use LP-filtered signal per phase |
| ScamSat | Acceleration | See PSD plot | Flagged if peak found |
| ScamSat | Temperature | No | Use LP-filtered signal |

The LP-filtered signals (trend preserved) are used in §5 for atmospheric analysis.


In [ ]:
# ── 4.4 Final filtered signals — VAMOS ───────────────────────────────────────
_apo_mf  = (vamos_drop['t_apogee_s']  - vamos['t_s'].iat[0]) / 60
_land_mf = (vamos_drop['t_landing_s'] - vamos['t_s'].iat[0]) / 60

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
fig.suptitle('VAMOS science — final filtered time series', fontweight='bold')

for ax, (sig, ylabel, col) in zip(axes, [
    (p_vf / 100, 'Pressure (hPa)',  'tab:blue'),
    (T_vf,       'Temperature (°C)','tab:red'),
]):
    ax.plot(t_vu/60, sig, lw=0.9, color=col)
    ax.axvspan(_apo_mf, _land_mf, color='gold', alpha=0.3, label='Drop phase')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8, loc='upper right')

_co2_t = vamos_co2['t_rel'].values / 60
axes[2].plot(_co2_t, vamos_co2['co2_ppm'].values, lw=0.7, color='tab:brown', alpha=0.8)
axes[2].axhline(CO2_BACKGROUND_PPM, ls='--', color='gray', lw=1.2, label=f'Background {CO2_BACKGROUND_PPM} ppm')
axes[2].axvspan(_apo_mf, _land_mf, color='gold', alpha=0.3, label='Drop phase')
axes[2].set(ylabel='CO₂ (ppm)', xlabel='Time (min)')
axes[2].legend(fontsize=8)
plt.tight_layout()
save_figure(fig, 'signal_processing/fig_vamos_final.png', strip_text=False)
plt.show()


In [ ]:
# ── 4.4 Final filtered signals — ScamSat ─────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
fig.suptitle('ScamSat — filtered time series', fontweight='bold')
axes[0].plot(_ta_u/3600, _aa_f, lw=0.7, color='tab:purple')
axes[0].set_ylabel('Acceleration (g)')
axes[1].plot(_tt_u/3600, _aT_f, lw=0.7, color='tab:red')
axes[1].set(ylabel='Temperature (°C)', xlabel='Time (h)')
plt.tight_layout()
save_figure(fig, 'signal_processing/fig_scamsat_final.png', strip_text=False)
plt.show()


---
## 5. Atmospheric Time Series Analysis

With pre-processed signals in hand, we address the atmospheric component of the research question.  
We use:
- **GRASP** for high-frequency descent profiles (temperature, pressure, PM₂.₅/PM₁₀ at ~38 Hz, LP-filtered).
- **VAMOS science** for CO₂ and a longer-context T/P record including the drop window.
- **VAMOS wind** for the wind speed/direction profile during the descent.
- **OBAMA** as independent T/P/RH spot-check (not time-synchronised; order-of-magnitude only).
- **ISA standard atmosphere** as the theoretical vertical reference.


### 5.1 Temperature and Pressure — Vertical Profiles

In [ ]:
# ── 5.1-pre  Combined temperature profiles — VAMOS & ScamSat ─────────────────

# --- ScamSat: interpolate pressure and baro altitude onto filtered-T time grid
_scam_h_baro = np.interp(
    _tt_u,
    scam_press['t_rel'].values,
    isa_alt(scam_press['press_Pa'].values),   # barometric altitude (m ASL, ISA)
)
_scam_p_hPa = np.interp(
    _tt_u,
    scam_press['t_rel'].values,
    scam_press['press_Pa'].values,
) / 100.0

# Keep only the airborne descent window (400–1 300 m ASL covers the full flight)
_sc_mask = (_scam_h_baro > 400) & (_scam_h_baro < 1300)

# --- VAMOS: drop window on the filtered signal already on grid t_vu / p_vf ---
_t0 = vamos_drop['t_apogee_s']  - vamos['t_s'].iat[0]
_t1 = vamos_drop['t_landing_s'] - vamos['t_s'].iat[0]
_vm = (t_vu >= _t0) & (t_vu <= _t1)

_T_vm  = T_vf[_vm]
_h_vm  = isa_alt(p_vf[_vm])      # barometric altitude (m ASL)
_p_vm  = p_vf[_vm] / 100.0       # hPa

# --- ISA reference line ---
_h_isa_ref = np.linspace(400, 1300, 300)
_T_isa_ref = isa_temp_c(_h_isa_ref)
_p_isa_ref = isa_press(_h_isa_ref) / 100.0

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(13, 7))
fig.suptitle('Temperature profiles — VAMOS & ScamSat (LP-filtered, barometric altitude)',
             fontweight='bold')

# Left panel: T vs barometric altitude
ax = axes[0]
ax.plot(_T_vm, _h_vm,
        lw=1.4, color='tab:orange', alpha=0.85, label='VAMOS drop (filtered)')
ax.plot(_aT_f[_sc_mask], _scam_h_baro[_sc_mask],
        lw=1.0, color='tab:red',    alpha=0.75, label='ScamSat (filtered)')
ax.plot(_T_isa_ref, _h_isa_ref,
        lw=2.0, color='black', ls='--', label='ISA')
ax.set(xlabel='Temperature (°C)', ylabel='Barometric altitude (m ASL)',
       title='T vs altitude')
ax.legend(fontsize=8)

# Right panel: T vs pressure  (pressure as vertical coordinate — no datum issue)
ax = axes[1]
ax.plot(_T_vm, _p_vm,
        lw=1.4, color='tab:orange', alpha=0.85, label='VAMOS drop (filtered)')
ax.plot(_aT_f[_sc_mask], _scam_p_hPa[_sc_mask],
        lw=1.0, color='tab:red',    alpha=0.75, label='ScamSat (filtered)')
ax.plot(_T_isa_ref, _p_isa_ref,
        lw=2.0, color='black', ls='--', label='ISA')
ax.invert_yaxis()
ax.set(xlabel='Temperature (°C)', ylabel='Pressure (hPa)',
       title='T vs pressure')
ax.legend(fontsize=8)

plt.tight_layout()
save_figure(fig, 'cross_dataset/fig_T_profiles_vamos_scamsat.png', strip_text=False)
plt.show()


In [ ]:
# ── 5.1 GRASP filtered vertical profile ──────────────────────────────────────
_t_gu, _p_gu = resample_uniform(grasp_clean['t_rel'].values, grasp_clean['press_Pa'].values, float(FS_G))
_,     _T_gu = resample_uniform(grasp_clean['t_rel'].values, grasp_clean['temp_C'].values,  float(FS_G))
_p_gf = butter_lowpass(_p_gu, cutoff_hz=0.5, fs=float(FS_G), order=4)
_T_gf = butter_lowpass(_T_gu, cutoff_hz=0.3, fs=float(FS_G), order=4)
_h_gf = isa_alt(_p_gf)   # barometric altitude from filtered pressure

# ISA
_h_isa = np.linspace(300, 1600, 300)
_T_isa = isa_temp_c(_h_isa)
_p_isa = isa_press(_h_isa) / 100.0

# VAMOS drop window
_vd    = vamos[drop_mask]
_h_vd  = isa_alt(_vd['press_hPa'].values * 100.0)

# OBAMA
_ob    = obama.dropna(subset=['first_pstat_avg_hPa','first_temp_avg_C'])
_h_ob1 = isa_alt(_ob['first_pstat_avg_hPa'].values * 100.0)
_h_ob2 = isa_alt(_ob['second_pstat_avg_hPa'].values * 100.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
fig.suptitle('Vertical profiles — temperature and pressure', fontweight='bold')

ax = axes[0]
ax.plot(_T_gf, _h_gf, lw=0.8, color='tab:red',    alpha=0.7, label='GRASP (LP filtered)')
ax.plot(_vd['temp_C'].values, _h_vd, lw=1.2, color='tab:orange', alpha=0.8, label='VAMOS (drop)')
ax.plot(_T_isa, _h_isa, lw=2.0, color='black', ls='--', label='ISA')
ax.scatter(_ob['first_temp_avg_C'],  _h_ob1, s=20, color='tab:blue',   alpha=0.7, label='OBAMA S1')
ax.scatter(_ob['second_temp_avg_C'], _h_ob2, s=20, color='tab:purple', alpha=0.7, label='OBAMA S2')
ax.set(xlabel='Temperature (°C)', ylabel='Barometric altitude (m ASL)', title='Temperature profile')
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(_p_gf/100, _h_gf, lw=0.8, color='tab:blue', alpha=0.7, label='GRASP (LP filtered)')
ax.plot(_vd['press_hPa'].values, _h_vd, lw=1.2, color='tab:cyan', alpha=0.8, label='VAMOS (drop)')
ax.plot(_p_isa, _h_isa, lw=2.0, color='black', ls='--', label='ISA')
ax.scatter(_ob['first_pstat_avg_hPa'],  _h_ob1, s=20, color='tab:blue',   alpha=0.7, label='OBAMA S1')
ax.scatter(_ob['second_pstat_avg_hPa'], _h_ob2, s=20, color='tab:purple', alpha=0.7, label='OBAMA S2')
ax.set(xlabel='Pressure (hPa)', ylabel='Barometric altitude (m ASL)', title='Pressure profile')
ax.legend(fontsize=8)

plt.tight_layout()
save_figure(fig, 'cross_dataset/fig_vertical_profiles.png', strip_text=False)
plt.show()

_lapse = np.polyfit(_h_gf, _T_gf, 1)[0] * 1000
_bias  = _T_gf.mean() - isa_temp_c(_h_gf).mean()
print(f'GRASP lapse rate: {_lapse:+.2f} K km⁻¹  (ISA: −6.5 K km⁻¹)')
print(f'Mean T anomaly vs ISA: {_bias:+.1f} K')


### 5.2 Particulate Matter — PM₂.₅ and PM₁₀

WHO 2021 Air Quality Guidelines: PM₂.₅ < 15 µg m⁻³, PM₁₀ < 45 µg m⁻³ (24-h mean).

In [ ]:
# ── 5.2 PM analysis ───────────────────────────────────────────────────────────
_pm25 = grasp_clean['pm25'].values
_pm10 = grasp_clean['pm10'].values
_h    = grasp_clean['alt_baro'].values
_tg   = grasp_clean['t_rel'].values

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('GRASP — PM₂.₅ and PM₁₀', fontweight='bold')

ax = axes[0,0]
ax.scatter(_pm25, _h, s=4, alpha=0.5, color='tab:orange', label='PM₂.₅')
ax.scatter(_pm10, _h, s=4, alpha=0.5, color='tab:brown',  label='PM₁₀')
ax.axvline(WHO_PM25_24H, color='tab:orange', lw=1.5, ls='--', label=f'WHO PM₂.₅ = {WHO_PM25_24H}')
ax.axvline(WHO_PM10_24H, color='tab:brown',  lw=1.5, ls='--', label=f'WHO PM₁₀ = {WHO_PM10_24H}')
ax.set(xlabel='µg m⁻³', ylabel='Baro altitude (m ASL)', title='PM vs altitude')
ax.legend(fontsize=7)

ax = axes[0,1]
ax.plot(_tg, _pm25, lw=0.8, color='tab:orange', label='PM₂.₅')
ax.plot(_tg, _pm10, lw=0.8, color='tab:brown',  label='PM₁₀')
ax.axhline(WHO_PM25_24H, color='tab:orange', lw=1.5, ls='--')
ax.axhline(WHO_PM10_24H, color='tab:brown',  lw=1.5, ls='--')
ax.set(xlabel='Time (s)', ylabel='µg m⁻³', title='PM vs time')
ax.legend(fontsize=8)

for ax, data, lbl, lim, col in [
    (axes[1,0], _pm25, 'PM₂.₅', WHO_PM25_24H, 'tab:orange'),
    (axes[1,1], _pm10, 'PM₁₀',  WHO_PM10_24H, 'tab:brown'),
]:
    ax.hist(data, bins=40, color=col, alpha=0.7, edgecolor='white')
    ax.axvline(lim, color='red', lw=1.5, ls='--', label=f'WHO = {lim}')
    ax.set(xlabel=f'{lbl} (µg m⁻³)', ylabel='Count', title=f'{lbl} distribution')
    ax.legend(fontsize=8)

plt.tight_layout()
save_figure(fig, 'signal_processing/fig_pm_analysis.png', strip_text=False)
plt.show()
print(f'PM₂.₅ > WHO limit: {np.mean(_pm25 > WHO_PM25_24H)*100:.1f}% of samples')
print(f'PM₂.₅ median: {np.median(_pm25):.1f}  max: {np.max(_pm25):.1f}  µg m⁻³')
print(f'PM₁₀  median: {np.median(_pm10):.1f}  max: {np.max(_pm10):.1f}  µg m⁻³')


### 5.3 CO₂ — VAMOS

NDIR sensor, warm-up zeros excluded. Global atmospheric background: 420 ppm (NOAA, 2024).

In [ ]:
# ── 5.3 CO₂ analysis ─────────────────────────────────────────────────────────
_co2_t = vamos_co2['t_rel'].values / 60
_co2_v = vamos_co2['co2_ppm'].values
_co2_h = isa_alt(vamos_co2['press_hPa'].values * 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('VAMOS — CO₂ (warm-up removed)', fontweight='bold')

ax = axes[0]
ax.plot(_co2_t, _co2_v, lw=0.7, color='tab:brown', alpha=0.8)
ax.axvspan(_apo_mf, _land_mf, color='gold', alpha=0.3, label='Drop phase')
ax.axhline(CO2_BACKGROUND_PPM, ls='--', color='gray', lw=1.4, label=f'Background {CO2_BACKGROUND_PPM} ppm')
ax.set(xlabel='Time (min)', ylabel='CO₂ (ppm)', title='CO₂ vs time')
ax.legend(fontsize=8)

ax = axes[1]
sc = ax.scatter(_co2_v, _co2_h, s=3, alpha=0.3, c=_co2_t, cmap='plasma')
ax.axvline(CO2_BACKGROUND_PPM, ls='--', color='gray', lw=1.4, label=f'Background {CO2_BACKGROUND_PPM} ppm')
ax.set(xlabel='CO₂ (ppm)', ylabel='Baro altitude (m ASL)', title='CO₂ vs altitude')
ax.legend(fontsize=8)
plt.colorbar(sc, ax=ax, label='Time (min)')

plt.tight_layout()
save_figure(fig, 'signal_processing/fig_co2_analysis.png', strip_text=False)
plt.show()
print(f'CO₂ median: {np.median(_co2_v):.0f} ppm | max: {np.max(_co2_v):.0f} ppm')
print(f'Above background: {np.mean(_co2_v > CO2_BACKGROUND_PPM)*100:.1f}% of samples')


### 5.4 Wind — VAMOS

Full mission time series and vertical profile during the drop phase.

In [ ]:
# ── 5.4 Wind analysis ────────────────────────────────────────────────────────
_drop_w = wind[wind['t_s'].between(t_apo - 10, t_land + 10)].copy()
_h_drop = np.interp(_drop_w['t_s'].values, vamos['t_s'].values, vamos_drop['h_agl'])

fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.45)
fig.suptitle('VAMOS wind — time series and vertical profile', fontweight='bold')

_twh = wind['t_rel'].values / 3600
_d0h = (t_apo  - wind['t_s'].iat[0]) / 3600
_d1h = (t_land - wind['t_s'].iat[0]) / 3600

ax = fig.add_subplot(gs[0, :2])
ax.plot(_twh, wind['wind_spd_vec'], lw=0.6, color='tab:cyan', alpha=0.8)
ax.axvspan(_d0h, _d1h, color='gold', alpha=0.3, label='Drop phase')
ax.set(xlabel='Time (h)', ylabel='Wind speed |vec| (m s⁻¹)', title='Wind speed — full mission')
ax.legend(fontsize=8)

ax = fig.add_subplot(gs[1, :2])
ax.scatter(_twh, wind['wind_dir'], s=1.5, alpha=0.4, color='tab:purple')
ax.axvspan(_d0h, _d1h, color='gold', alpha=0.3)
ax.set(xlabel='Time (h)', ylabel='Wind direction (°)', title='Wind direction — full mission')

ax = fig.add_subplot(gs[:, 2])
sc = ax.scatter(_drop_w['wind_spd_vec'].values, _h_drop,
                s=25, alpha=0.7, c=_drop_w['t_s'].values, cmap='viridis')
ax.set(xlabel='Wind speed |vec| (m s⁻¹)', ylabel='Altitude AGL (m)', title='Wind vs altitude (drop)')
plt.colorbar(sc, ax=ax, label='Time (s)')

plt.tight_layout()
save_figure(fig, 'flight_dynamics/fig_wind_profile.png', strip_text=False)
plt.show()
print(f'Drop-phase wind — mean: {_drop_w["wind_spd_vec"].mean():.2f} m/s | max: {_drop_w["wind_spd_vec"].max():.2f} m/s')


---
## 6. Conclusions

### 6.1 Spectral analysis

| Dataset | Signal | Oscillatory artefact | Outcome |
|---------|--------|---------------------|---------|
| VAMOS science | Pressure + temperature | None detected | Broadband noise only; signal dominated by atmospheric trend |
| VAMOS wind | Wind speed + acceleration | None detected | Phase-dependent energy; no narrow pendulum peak |
| ScamSat | Acceleration | See §4.3 PSD | Flag if peak present at 0.3–1 Hz |

No evidence of persistent parachute pendulum oscillation was found in any of the three analysed datasets.  The descent was mechanically stable.

### 6.2 Atmospheric observations

| Variable | Observation | Physical interpretation |
|----------|------------|------------------------|
| Temperature lapse rate | Sub-standard; ~+7–8 K above ISA at sampled altitudes | Stable winter boundary layer; possible temperature inversion |
| Pressure | Consistent with ISA; small positive continental bias | Synoptically near-normal surface pressure |
| PM₂.₅ | Significant fraction above WHO 24-h limit (15 µg m⁻³) near ground | Aerosol accumulation in stable BL; anthropogenic sources |
| PM₁₀ | Generally below WHO limit (45 µg m⁻³) | Larger particles settle more efficiently |
| CO₂ | Median above global background (420 ppm) | Local emissions (traffic, combustion near Dübendorf) |
| Wind | Low speeds during drop; relatively stable direction | Weak synoptic flow; no strong turbulence |

### 6.3 Limitations

- **GRASP** (176 s, descent only) was excluded from the spectral pipeline; a longer hover phase would improve frequency resolution.
- **OBAMA** (31 rows) provides only a coarse sanity check; simultaneous synchronised recording would allow proper inter-sensor validation.
- The VAMOS `tumbling` flag was non-functional; a reliable IMU tumbling indicator would improve artefact characterisation.
- Future work: include UWYO radiosonde and ERA5 reanalysis for synoptic context; apply notch filtering if pendulum peaks are confirmed in a future flight.
